# 🐼 Panda AI — One-Click Deploy
Deploy the full Panda AI gateway (API + Dashboard) on Colab.
**Runtime → Run all (Ctrl+F9)**

## 1️⃣ Install

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs > /dev/null 2>&1
!pip install -q -r requirements.txt --root-user-action=ignore 2>&1 | grep -v WARNING | tail -2
!patchright install chromium 2>&1 | tail -1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cf && mv /tmp/cf /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import subprocess, sys; print(f'✅ Python {sys.version.split()[0]} | Node {subprocess.check_output(["node","--version"]).decode().strip()}')

## 2️⃣ Clone & Config

In [ ]:
import os, secrets
os.chdir('/')
!rm -rf /content/Panda-Ai
!git clone -q https://github.com/ferelking242/Panda-Ai.git /content/Panda-Ai
os.chdir('/content/Panda-Ai')
api_token = 'pnd_' + secrets.token_hex(16)
with open('.env', 'w') as f: f.write(f'PROVIDER=chatgpt\nHEADLESS=true\nAPI_HOST=0.0.0.0\nAPI_PORT=8000\nAPI_TOKEN={api_token}\nPOOL_SIZE=1\nLOG_LEVEL=INFO\n')
print(f'✅ Ready | 🔑 Token: {api_token}')

## 3️⃣ Build Dashboard

In [ ]:
os.chdir('/content/Panda-Ai/dashboard')
!npm install --no-audit --no-fund --silent 2>&1 | tail -1
!npm run build 2>&1 | tail -3
os.chdir('/content/Panda-Ai')
print('✅ Dashboard built')

## 4️⃣ Start

In [ ]:
import subprocess, time, os, sys

# Kill ALL processes on ports 8000 and 5000 (not by name — name-based pkill misses python -m)
!fuser -k 8000/tcp 2>/dev/null || true
!fuser -k 5000/tcp 2>/dev/null || true
time.sleep(2)

env8k = {**os.environ, 'PYTHONUNBUFFERED': '1', 'API_TOKEN': api_token}

api_proc = subprocess.Popen(
    [sys.executable, '-m', 'src.api.server'],
    cwd='/content/Panda-Ai', env=env8k,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(f'🚀 API starting (PID {api_proc.pid})...')

dash_proc = subprocess.Popen(
    ['node', 'server.js'],
    cwd='/content/Panda-Ai/dashboard',
    env={**env8k, 'PORT': '5000', 'API_ORIGIN': 'http://127.0.0.1:8000', 'NODE_ENV': 'production'},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(f'📊 Dashboard starting (PID {dash_proc.pid})...')

import urllib.request
for i in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/healthz', timeout=3)
        print(f'✅ API healthy')
        break
    except: pass

## 5️⃣ Expose URLs

In [ ]:
import re, threading
urls = {}
def tunnel(port, name):
    p = subprocess.Popen(['cloudflared','tunnel','--url',f'http://localhost:{port}'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if m: urls[name] = m.group(0); print(f'  🔗 {name}: {m.group(0)}'); break
threading.Thread(target=tunnel, args=(8000,'API'), daemon=True).start()
threading.Thread(target=tunnel, args=(5000,'Dashboard'), daemon=True).start()
for _ in range(30):
    time.sleep(1)
    if len(urls) >= 2: break
print(f'\n═══════════════════════════════════════════')
print(f'  🐼 PANDA AI DEPLOYED')
print(f'═══════════════════════════════════════════')
if 'API' in urls: print(f'  🤖 API:      {urls["API"]}/v1')
if 'Dashboard' in urls: print(f'  📊 Dashboard: {urls["Dashboard"]}')
print(f'  🔑 Token: {api_token}')
print(f'═══════════════════════════════════════════')

## 📋 Quick Test

In [ ]:
import urllib.request, json
base = 'http://127.0.0.1:8000'

health = json.loads(urllib.request.urlopen(f'{base}/healthz').read())
print(f'✅ Health: {health}')

try:
    req = urllib.request.Request(f'{base}/v1/models', headers={'Authorization': f'Bearer {api_token}'})
    models = json.loads(urllib.request.urlopen(req).read())
    print(f'✅ Models: {[m["id"] for m in models["data"][:5]]}')
except urllib.error.HTTPError as e:
    body = e.read().decode()
    print(f'❌ /v1/models: {e.code} — {body[:200]}')
    # Debug: check what token the server expects
    try:
        from src.config import Config
        print(f'   Server API_TOKEN = {repr(Config.API_TOKEN[:12])}...')
        print(f'   Client sent      = {repr(api_token[:12])}...')
        print(f'   Match: {Config.API_TOKEN == api_token}')
    except Exception as ex:
        print(f'   Config import failed: {ex}')